# Introduction to Lasso Regression and Ames Housing Dataset

## The Dataset: Ames Housing
The Ames Housing dataset is a comprehensive alternative to the classic Boston Housing dataset. It contains 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa. The target variable we want to predict is the `SalePrice`. With its high dimensionality and a rich mix of numerical and categorical features, it serves as an excellent benchmark for data cleaning, feature engineering, and advanced regression techniques.

## The Model: Lasso Regression
Lasso (Least Absolute Shrinkage and Selection Operator) is a linear regression model that incorporates **L1 regularization**. 
Unlike standard ordinary least squares (OLS) regression, Lasso applies a penalty equal to the absolute value of the coefficients' magnitude. This key mechanism allows Lasso to perform **automatic feature selection** by shrinking the weights of less relevant features exactly to zero. As a result, it yields a simpler, more robust, and highly interpretable model, which is especially useful when dealing with datasets that have a large number of features.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

In [ ]:
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df = ames.frame
df.columns

## Data Cleaning: Intelligent Missing Value Imputation

In this section, we handle missing data (`NaN`) using three distinct strategies tailored to the specific context of the Ames dataset:

1. **Context-Aware Imputation ("None"):** For certain structural features (e.g., `PoolQC`, `GarageType`, `BsmtQual`), a missing value is not an error but indicates the physical absence of that amenity (e.g., the house has no pool or no basement). We iterate through these specific columns and replace `NaN` with the string `"None"`. For pandas `category` data types, we explicitly add `"None"` to the allowed categories first.
2. **Numerical Imputation (Median):** For all numeric columns (integers and floats), missing values are filled using the median of the respective column. The median is preferred over the mean as it is more robust to outliers (e.g., extremely large houses).
3. **Categorical Imputation (Mode):** For any remaining categorical or text-based features where a missing value truly represents missing data, we fill the gaps using the mode (the most frequent value) of that specific column.

In [ ]:
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

for col in cols_none:
    if col in df.columns:
        if df[col].dtype.name == 'category':
            df[col] = df[col].cat.add_categories("None")
        df[col] = df[col].fillna("None")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object', 'category']).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [ ]:
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

if 'Id' in df.columns:
    df = df.drop('Id', axis=1)

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop('SalePrice', axis=1)
y = df_encoded['SalePrice']

print(f'Cols before transform: {df.shape[1]}')
print(f'Cols after One-Hot Encoding: {X.shape[1]}')